# Experiment: Trim QHARM TS Database to Released 8980 Set

Objective:
- Keep the released 8980 transition-state combinations defined by `Data/TS/Borane_all.csv`.
- Remove only the extra TS rows from `BorylXAT-DB_qh_update.db`, while leaving all non-TS structures unchanged.
- Write the result to a new database file instead of modifying the original QHARM database in place.


In [ ]:
from __future__ import annotations

import shutil
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display

KEY_COLUMNS = ["B_Index", "N_Index", "Cl_Index"]
OUTPUT_DB_NAME = "BorylXAT-DB_qh_update_trimmed_to_8980.db"
OVERWRITE_OUTPUT = True


def find_repo_root(start: Path) -> Path:
    markers = [".git", "README.md", "pyproject.toml"]
    for candidate in (start.resolve(), *start.resolve().parents):
        if any((candidate / marker).exists() for marker in markers):
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
QHARM_DB_PATH = REPO_ROOT / "BorylXAT-DB_qh_update.db"
RELEASE_TS_CSV_PATH = REPO_ROOT / "Data" / "TS" / "Borane_all.csv"
OUTPUT_DB_PATH = REPO_ROOT / OUTPUT_DB_NAME

assert QHARM_DB_PATH.exists(), QHARM_DB_PATH
assert RELEASE_TS_CSV_PATH.exists(), RELEASE_TS_CSV_PATH

print(f"Repository root: {REPO_ROOT}")
print(f"Source QHARM DB: {QHARM_DB_PATH.name}")
print(f"Released TS CSV : {RELEASE_TS_CSV_PATH.relative_to(REPO_ROOT)}")
print(f"Output DB       : {OUTPUT_DB_PATH.name}")


## Plan

- Build the released TS key set from `Borane_all.csv` using `(B_Index, N_Index, Cl_Index)`.
- Read every TS row from the QHARM database together with its `systems.id`.
- Identify the TS rows present in the QHARM database but absent from the released CSV.
- Copy the QHARM database, delete those extra TS rows from all ASE-linked tables, and validate that the trimmed database contains exactly 8980 TS rows.


In [ ]:
def load_release_ts_keys(csv_path: Path) -> pd.DataFrame:
    release_df = pd.read_csv(csv_path, usecols=KEY_COLUMNS)
    release_df = release_df.drop_duplicates().sort_values(KEY_COLUMNS).reset_index(drop=True)
    return release_df


def load_ts_rows(db_path: Path) -> pd.DataFrame:
    query = """
    SELECT
        s.id AS system_id,
        MAX(CASE WHEN tkv.key = 'key' THEN tkv.value END) AS structure_key,
        CAST(MAX(CASE WHEN nkv.key = 'B_id' THEN nkv.value END) AS INTEGER) AS B_Index,
        CAST(MAX(CASE WHEN nkv.key = 'LB_id' THEN nkv.value END) AS INTEGER) AS N_Index,
        CAST(MAX(CASE WHEN nkv.key = 'Cl_id' THEN nkv.value END) AS INTEGER) AS Cl_Index
    FROM systems AS s
    LEFT JOIN text_key_values AS tkv ON s.id = tkv.id
    LEFT JOIN number_key_values AS nkv ON s.id = nkv.id
    WHERE s.id IN (
        SELECT id
        FROM text_key_values
        WHERE key = 'category' AND value = 'ts'
    )
      AND (nkv.key IN ('B_id', 'LB_id', 'Cl_id') OR nkv.key IS NULL)
      AND (tkv.key = 'key' OR tkv.key IS NULL)
    GROUP BY s.id
    ORDER BY s.id
    """
    with sqlite3.connect(db_path) as connection:
        ts_df = pd.read_sql_query(query, connection)
    return ts_df


def count_categories(db_path: Path) -> pd.Series:
    query = """
    SELECT value AS category, COUNT(*) AS count
    FROM text_key_values
    WHERE key = 'category'
    GROUP BY value
    ORDER BY value
    """
    with sqlite3.connect(db_path) as connection:
        counts = pd.read_sql_query(query, connection)
        total_systems = connection.execute("SELECT COUNT(*) FROM systems").fetchone()[0]
    series = counts.set_index("category")["count"].sort_index()
    series.loc["__total_systems__"] = total_systems
    return series


release_keys_df = load_release_ts_keys(RELEASE_TS_CSV_PATH)
qharm_ts_df = load_ts_rows(QHARM_DB_PATH)
source_category_counts = count_categories(QHARM_DB_PATH)

extra_ts_df = (
    qharm_ts_df.merge(release_keys_df, on=KEY_COLUMNS, how="left", indicator=True)
    .loc[lambda df: df["_merge"] == "left_only", ["system_id", "structure_key", *KEY_COLUMNS]]
    .sort_values(KEY_COLUMNS + ["system_id"])
    .reset_index(drop=True)
)

summary = {
    "released_ts_keys": int(len(release_keys_df)),
    "qharm_ts_rows": int(len(qharm_ts_df)),
    "extra_ts_rows": int(len(extra_ts_df)),
    "expected_trimmed_ts_rows": int(len(qharm_ts_df) - len(extra_ts_df)),
}
summary


In [ ]:
assert len(release_keys_df) == 8980, len(release_keys_df)
assert len(qharm_ts_df) == 9237, len(qharm_ts_df)
assert len(extra_ts_df) == 257, len(extra_ts_df)
assert summary["expected_trimmed_ts_rows"] == 8980

display(source_category_counts.rename("source_count").to_frame())
display(extra_ts_df.head(12))
display(extra_ts_df["B_Index"].value_counts().head(10).rename("extra_ts_count_by_B"))


## Trim the copied database

The cell below makes a copy of the QHARM database and deletes only the extra TS `systems.id` rows, plus their linked entries in the ASE helper tables.


In [ ]:
def delete_system_ids(db_path: Path, system_ids: list[int]) -> None:
    if not system_ids:
        return

    placeholders = ", ".join("?" for _ in system_ids)
    delete_tables = ["species", "keys", "text_key_values", "number_key_values", "systems"]

    with sqlite3.connect(db_path) as connection:
        cursor = connection.cursor()
        for table in delete_tables:
            cursor.execute(f"DELETE FROM {table} WHERE id IN ({placeholders})", system_ids)
        connection.commit()
        connection.execute("VACUUM")


if OUTPUT_DB_PATH.exists():
    if OVERWRITE_OUTPUT:
        OUTPUT_DB_PATH.unlink()
    else:
        raise FileExistsError(
            f"Output database already exists: {OUTPUT_DB_PATH}. Set OVERWRITE_OUTPUT = True to replace it."
        )

shutil.copy2(QHARM_DB_PATH, OUTPUT_DB_PATH)
delete_system_ids(OUTPUT_DB_PATH, extra_ts_df["system_id"].astype(int).tolist())

print(f"Trimmed database written to: {OUTPUT_DB_PATH}")


In [ ]:
trimmed_category_counts = count_categories(OUTPUT_DB_PATH)
trimmed_ts_df = load_ts_rows(OUTPUT_DB_PATH)

assert len(trimmed_ts_df) == 8980, len(trimmed_ts_df)
assert len(trimmed_ts_df) == len(release_keys_df)

trimmed_key_set = set(map(tuple, trimmed_ts_df[KEY_COLUMNS].to_numpy()))
release_key_set = set(map(tuple, release_keys_df[KEY_COLUMNS].to_numpy()))

assert trimmed_key_set == release_key_set
assert trimmed_category_counts.loc["ts"] == 8980
assert int(source_category_counts.loc["ts"] - trimmed_category_counts.loc["ts"]) == len(extra_ts_df)

for category, source_count in source_category_counts.items():
    if category in {"ts", "__total_systems__"}:
        continue
    assert int(trimmed_category_counts.loc[category]) == int(source_count), (category, source_count, trimmed_category_counts.loc[category])

result = pd.DataFrame(
    {
        "source_qharm": source_category_counts.astype(int),
        "trimmed_output": trimmed_category_counts.astype(int),
    }
)
result["difference"] = result["trimmed_output"] - result["source_qharm"]

display(result)
print("Trimmed TS rows match the released 8980-key set exactly.")


## Next steps

- Use `BorylXAT-DB_qh_update_trimmed_to_8980.db` for Figure 5A-style TS type counts when you want QHARM thermochemistry but only the released 8980 TS space.
- If needed, copy the same key-filter logic into a production script so this trimming step does not live only in a notebook.
